# XGBoost + MAE loss + geo cats + property_type + ratios + date_ordinal

Promoted recipe from the model-family-mae auto loop on branch `mlchallenge/may5`.

**Recipe** (promoted = simplified Stage 3 winner; only `max_depth=6` was load-bearing per the leave-one-out ablation in `iteration_log.md`):
- `XGBRegressor(objective="reg:absoluteerror", n_estimators=500, learning_rate=0.05, max_depth=6, min_child_weight=5, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist", enable_categorical=True)` &mdash; directly optimizes MAE
- All numeric & boolean columns from `dataset/train.json` (excluding `parcel_ids`, `transferred_parcel_ids` which coerce to inf-magnitude strings)
- Derived ratios: `built_per_premise`, `land_per_lot`, `commercial_share`, `apt_share`, `houses_per_premise`
- `date_ordinal` = days since 2014-01-01
- Three native XGBoost categoricals (`enable_categorical=True`): `commune_first` (first token of `commune_codes`), `cadastral_first` (first token of `cadastral_sections`), `property_type`
- Categorical levels fitted on training only; unseen test values fall back to NaN (XGBoost handles natively)

**Performance** (canonical 5-fold CV-MAE, full audit in `experiments/xgboost/results.tsv` and `iteration_log.md`):

| Stage | Metric | Value |
|---|---|---|
| HGB iter21 (prior global champion) | 5-fold CV-MAE | 50,934.78 |
| xgboost smoke `geo_signal` | 5-fold CV-MAE | 51,530.01 |
| xgboost deepen iter10 (added property_type cat) | 5-fold CV-MAE | 50,957.36 |
| xgboost tune iter9 W (multi-change winner) | 5-fold CV-MAE | 49,546.40 |
| **promoted simplified recipe (this notebook, seed=42)** | **5-fold CV-MAE** | **49,685.50** |
| promoted recipe confirm (seed=2026) | 5-fold CV-MAE | 49,827.48 |
| **2-seed mean** | **5-fold CV-MAE** | **49,756.49** |

That's a **~1,178 MAE win over the prior HGB champion** (above the pooled-std noise threshold of ~1,200), promoted only after a full leave-one-out ablation suite confirmed `max_depth=6` was the only load-bearing tune knob.

In [ ]:
import json
import zipfile
from pathlib import Path

import dagshub
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from xgboost import XGBRegressor

dagshub.init(repo_owner="abdul.aldalali", repo_name="ML_Challenge_ML_Class_DSS", mlflow=True)
mlflow.set_experiment("xgboost")

cwd = Path.cwd()
if (cwd / "dataset").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "dataset").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd.parent.parent / "dataset").exists():
    PROJECT_ROOT = cwd.parent.parent
else:
    raise FileNotFoundError("Could not find project root containing dataset/")

EXPERIMENT_NAME = "xgboost"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / EXPERIMENT_NAME
DATASET_DIR = PROJECT_ROOT / "dataset"

TARGET_KEY = "property_value"
VALIDATION_SIZE = 0.2
RANDOM_STATE = 42
CV_FOLDS = 5

ID_COLUMNS = {"parcel_ids", "transferred_parcel_ids"}
GEO_MULTIVALUE_COLUMNS = ("commune_codes", "cadastral_sections")
CATEGORICAL_FEATURES = ("commune_first", "cadastral_first", "property_type")
DERIVED_RATIO_COLUMNS = (
    "built_per_premise",
    "land_per_lot",
    "commercial_share",
    "apt_share",
    "houses_per_premise",
)
DATE_FEATURE_COLUMNS = ("date_ordinal",)
DATE_EPOCH = pd.Timestamp("2014-01-01")

XGB_PARAMS = dict(
    objective="reg:absoluteerror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    enable_categorical=True,
)

TRAIN_PATH = DATASET_DIR / "train.json"
TEST_PATH = DATASET_DIR / "test.json"
PREDICTION_JSON_PATH = EXPERIMENT_DIR / "predicted.json"
PREDICTION_ZIP_PATH = EXPERIMENT_DIR / "predicted.zip"

In [ ]:
def load_json(path):
    """Load a JSON file and return the parsed Python object."""
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def make_target_array(records):
    return np.array([record[TARGET_KEY] for record in records], dtype=np.float64)


def first_token(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    text = str(value).strip()
    if not text:
        return np.nan
    return text.split(",")[0].strip()


def add_geo_features(frame):
    frame = frame.copy()
    if "commune_codes" in frame.columns:
        frame["commune_first"] = frame["commune_codes"].map(first_token)
    else:
        frame["commune_first"] = np.nan
    if "cadastral_sections" in frame.columns:
        frame["cadastral_first"] = frame["cadastral_sections"].map(first_token)
    else:
        frame["cadastral_first"] = np.nan
    return frame


def safe_div(numerator, denominator):
    num = pd.to_numeric(numerator, errors="coerce")
    den = pd.to_numeric(denominator, errors="coerce")
    den = den.where(den > 0, np.nan)
    return num / den


def add_derived_ratios(frame):
    frame = frame.copy()
    frame["built_per_premise"] = safe_div(frame.get("built_area"), frame.get("num_premises"))
    frame["land_per_lot"] = safe_div(frame.get("land_area"), frame.get("num_lots"))
    frame["commercial_share"] = safe_div(frame.get("num_commercial"), frame.get("num_premises"))
    frame["apt_share"] = safe_div(frame.get("num_apartments"), frame.get("num_premises"))
    frame["houses_per_premise"] = safe_div(frame.get("num_houses"), frame.get("num_premises"))
    return frame


def add_date_ordinal(frame):
    frame = frame.copy()
    parsed = pd.to_datetime(frame.get("transaction_date"), errors="coerce")
    frame["date_ordinal"] = (parsed - DATE_EPOCH).dt.days.astype("Float64")
    return frame


def infer_numeric_features(records):
    """Pick raw columns whose values coerce to numeric, excluding target / IDs / cats."""
    frame = pd.DataFrame(records)
    features = []
    for column in frame.columns:
        if column == TARGET_KEY or column in ID_COLUMNS:
            continue
        if column in CATEGORICAL_FEATURES or column in GEO_MULTIVALUE_COLUMNS:
            continue
        values = pd.to_numeric(frame[column], errors="coerce")
        if values.notna().any():
            features.append(column)
    features.extend(DERIVED_RATIO_COLUMNS)
    features.extend(DATE_FEATURE_COLUMNS)
    return features


def fit_category_levels(records):
    frame = add_geo_features(pd.DataFrame(records))
    levels = {}
    for column in CATEGORICAL_FEATURES:
        if column in frame.columns:
            levels[column] = list(pd.Series(frame[column]).dropna().unique())
        else:
            levels[column] = []
    return levels


def make_feature_frame(records, numeric_columns, category_levels):
    frame = add_date_ordinal(add_derived_ratios(add_geo_features(pd.DataFrame(records))))
    numeric_part = (
        frame.reindex(columns=numeric_columns)
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )
    cat_part = pd.DataFrame(index=frame.index)
    for column in CATEGORICAL_FEATURES:
        levels = category_levels[column]
        series = frame[column] if column in frame.columns else pd.Series([np.nan] * len(frame), index=frame.index)
        cat_part[column] = pd.Categorical(series, categories=levels)
    return pd.concat([numeric_part, cat_part], axis=1)


def build_model():
    return XGBRegressor(**XGB_PARAMS)

In [ ]:
train_records = load_json(TRAIN_PATH)
test_records = load_json(TEST_PATH)

numeric_features = infer_numeric_features(train_records)
feature_columns = list(numeric_features) + list(CATEGORICAL_FEATURES)

train_split_records, validation_records = train_test_split(
    train_records,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)

with mlflow.start_run(run_name=EXPERIMENT_NAME):
    mlflow.log_param("model_type", "XGBRegressor")
    mlflow.log_param("validation_size", VALIDATION_SIZE)
    mlflow.log_param("random_state", RANDOM_STATE)
    for k, v in XGB_PARAMS.items():
        mlflow.log_param(k, v)
    mlflow.log_param("target", TARGET_KEY)
    mlflow.log_param("numeric_features", ",".join(numeric_features))
    mlflow.log_param("categorical_features", ",".join(CATEGORICAL_FEATURES))
    mlflow.log_param("derived_ratios", ",".join(DERIVED_RATIO_COLUMNS))
    mlflow.log_param("date_features", ",".join(DATE_FEATURE_COLUMNS))
    mlflow.log_metric("train_records", len(train_records))
    mlflow.log_metric("train_split_records", len(train_split_records))
    mlflow.log_metric("validation_records", len(validation_records))
    mlflow.log_metric("test_records", len(test_records))
    mlflow.log_metric("numeric_feature_count", len(numeric_features))
    mlflow.log_metric("categorical_feature_count", len(CATEGORICAL_FEATURES))
    mlflow.log_metric("feature_count", len(feature_columns))

    # Single 80/20 holdout (matches HGB notebook for cross-family comparability)
    holdout_levels = fit_category_levels(train_split_records)
    holdout_train_X = make_feature_frame(train_split_records, numeric_features, holdout_levels)
    holdout_val_X = make_feature_frame(validation_records, numeric_features, holdout_levels)
    holdout_model = build_model()
    holdout_model.fit(holdout_train_X, make_target_array(train_split_records))
    validation_predictions = holdout_model.predict(holdout_val_X)
    validation_targets = make_target_array(validation_records)
    validation_mae = mean_absolute_error(validation_targets, validation_predictions)
    validation_rmse = np.sqrt(mean_squared_error(validation_targets, validation_predictions))
    validation_r2 = r2_score(validation_targets, validation_predictions)

    mlflow.log_metric("validation_mae", validation_mae)
    mlflow.log_metric("validation_rmse", validation_rmse)
    mlflow.log_metric("validation_r2", validation_r2)

    # 5-fold CV-MAE -- the canonical workflow metric (matches scripts/eval_family.py)
    kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    fold_maes = []
    for fold, (tr_idx, vl_idx) in enumerate(kf.split(np.arange(len(train_records)))):
        fold_train_records = [train_records[i] for i in tr_idx]
        fold_val_records = [train_records[i] for i in vl_idx]
        fold_levels = fit_category_levels(fold_train_records)
        fold_train_X = make_feature_frame(fold_train_records, numeric_features, fold_levels)
        fold_val_X = make_feature_frame(fold_val_records, numeric_features, fold_levels)
        fold_model = build_model()
        fold_model.fit(fold_train_X, make_target_array(fold_train_records))
        fold_preds = fold_model.predict(fold_val_X)
        fold_mae = mean_absolute_error(make_target_array(fold_val_records), fold_preds)
        fold_maes.append(fold_mae)
        mlflow.log_metric(f"cv_fold_{fold}_mae", fold_mae)

    cv_mae = float(np.mean(fold_maes))
    cv_mae_std = float(np.std(fold_maes, ddof=0))
    mlflow.log_metric("cv_mae", cv_mae)
    mlflow.log_metric("cv_mae_std", cv_mae_std)

    # Train final model on full training data, predict test set
    full_levels = fit_category_levels(train_records)
    full_train_X = make_feature_frame(train_records, numeric_features, full_levels)
    test_X = make_feature_frame(test_records, numeric_features, full_levels)
    model = build_model()
    model.fit(full_train_X, make_target_array(train_records))
    test_predictions = model.predict(test_X)
    predictions = [{TARGET_KEY: float(value)} for value in test_predictions]

    PREDICTION_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(PREDICTION_JSON_PATH, "w", encoding="utf-8") as handle:
        json.dump(predictions, handle, indent=2)

    with zipfile.ZipFile(PREDICTION_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(PREDICTION_JSON_PATH, arcname="predicted.json")

    mlflow.log_metric("prediction_count", len(predictions))
    mlflow.sklearn.log_model(model, name=f"{EXPERIMENT_NAME}_model")
    mlflow.log_artifact(str(PREDICTION_JSON_PATH))
    mlflow.log_artifact(str(PREDICTION_ZIP_PATH))

print(f"Validation MAE (holdout): {validation_mae:,.2f}")
print(f"Validation RMSE (holdout): {validation_rmse:,.2f}")
print(f"Validation R2 (holdout): {validation_r2:.4f}")
print(f"5-fold CV MAE: {cv_mae:,.2f} (std {cv_mae_std:,.2f})")
print(f"Wrote {len(predictions)} predictions to {PREDICTION_ZIP_PATH}")

In [ ]:
{
    "experiment": EXPERIMENT_NAME,
    "primary_metric": "cv_mae",
    "cv_mae": cv_mae,
    "cv_mae_std": cv_mae_std,
    "validation_mae": validation_mae,
    "validation_rmse": validation_rmse,
    "validation_r2": validation_r2,
    "numeric_feature_count": len(numeric_features),
    "categorical_features": list(CATEGORICAL_FEATURES),
    "derived_ratios": list(DERIVED_RATIO_COLUMNS),
    "date_features": list(DATE_FEATURE_COLUMNS),
    "prediction_json": str(PREDICTION_JSON_PATH),
    "prediction_zip": str(PREDICTION_ZIP_PATH),
}